# Semantic Autoscaling: Test Results

This Google Colab notebook presents the verified simulation results for the semantic-archetype-aware autoscaling prototype.

**Claim under test:** at approximately constant aggregate request volume, conditioning capacity and cost-governance decisions on workload archetype reduces latency-SLA violations and improves budget-aware behavior compared with the aggregate-only baseline.

The notebook reads committed CSV and JSON artifacts. It does not call AWS and does not fabricate new measurements.

## Colab setup

Upload the repository ZIP to Colab, or clone the GitHub repository. Set `REPO` to the extracted repository path if needed.

In [ ]:
# Option A: clone the repository
!git clone -q https://github.com/Rohithpranov07/Cloud.git

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

REPO = Path('/content/Cloud')
DATA = REPO / 'data'
assert DATA.exists(), f'Could not find data directory: {DATA}'
sns.set_theme(style='whitegrid', palette='colorblind')
pd.set_option('display.max_columns', None)
print(f'Repository: {REPO}')
print(f'Artifacts: {len(list(DATA.iterdir()))} files')

In [ ]:
evaluation = pd.read_csv(DATA / 'evaluation_results.csv')
multiseed = pd.read_csv(DATA / 'multiseed_results.csv')
sensitivity = pd.read_csv(DATA / 'sensitivity.csv')
latency = json.loads((DATA / 'latency_summary.json').read_text())

print('Evaluation rows:', len(evaluation))
print('Multiseed rows:', len(multiseed))
print('Sensitivity rows:', len(sensitivity))
display(evaluation.head())

## 1. Headline result

The paired multi-seed evaluation uses 30 seeds. The sizing rule is reported explicitly because `round` is the frozen default and `ceil` is the sensitivity option.

In [ ]:
def headline(sizing):
    result = latency[sizing]
    rows = []
    for metric, values in result.items():
        rows.append({
            'sizing': sizing,
            'metric': metric,
            'n': values['n'],
            'baseline_median': values['median_baseline'],
            'aware_median': values['median_aware'],
            'paired_difference': values['median_paired_diff'],
            'p_value': values['p_value'],
            'rank_biserial_r': values['rank_biserial_r'],
        })
    return pd.DataFrame(rows)

headline_table = pd.concat([headline('round'), headline('ceil')], ignore_index=True)
display(headline_table)

plot_data = headline_table[headline_table['metric'].isin(['sla_violation_rate', 'p95_response_s'])].copy()
plot_data['metric'] = plot_data['metric'].map({'sla_violation_rate': 'SLA violation rate', 'p95_response_s': 'p95 response (s)'})
plot_data = plot_data.melt(id_vars=['sizing', 'metric'], value_vars=['baseline_median', 'aware_median'], var_name='policy', value_name='value')
plot_data['policy'] = plot_data['policy'].map({'baseline_median': 'Baseline', 'aware_median': 'Archetype-aware'})
g = sns.catplot(data=plot_data, x='sizing', y='value', hue='policy', col='metric', kind='bar', height=4, aspect=1.1)
g.set_axis_labels('Sizing rule', 'Median value')
g.set_titles('{col_name}')
plt.show()

### Interpretation

- The baseline is aggregate-rate-only and cannot distinguish expensive agentic work from short conversational work.
- The aware controller forecasts each archetype independently and allocates capacity to the primitive that needs it.
- The p-values and effect sizes are reported measurements, not correctness gates. Re-run the repository tests separately for acceptance.

## 2. Per-seed latency and SLA comparison

In [ ]:
round_results = multiseed[multiseed['sizing'].eq('round')]
ceil_results = multiseed[multiseed['sizing'].eq('ceil')]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for frame, label, color in [(round_results, 'round', '#fc5000'), (ceil_results, 'ceil', '#524ae9')]:
    axes[0].scatter(frame['baseline_p95_response_s'], frame['aware_p95_response_s'], label=label, alpha=.7, color=color)
max_value = axes[0].get_xlim()[1]
axes[0].plot([0, max_value], [0, max_value], '--', color='#070607', linewidth=1)
axes[0].set_title('p95 response: aware vs baseline')
axes[0].set_xlabel('Baseline p95 response (s)')
axes[0].set_ylabel('Aware p95 response (s)')
axes[0].legend(title='Sizing')

sns.boxplot(data=multiseed.melt(id_vars=['seed', 'sizing'], value_vars=['baseline_sla_violation_rate', 'aware_sla_violation_rate'], var_name='policy', value_name='rate'), x='sizing', y='rate', hue='policy', ax=axes[1])
axes[1].set_title('SLA violation rate across seeds')
axes[1].set_xlabel('Sizing rule')
axes[1].set_ylabel('Violation rate')
axes[1].legend(title='Policy')
plt.tight_layout()
plt.show()

display(multiseed.groupby('sizing')[['baseline_p95_response_s', 'aware_p95_response_s', 'baseline_sla_violation_rate', 'aware_sla_violation_rate']].median())

## 3. Constant-volume composition shift and controller response

This plot uses the evaluation trace to show the central mechanism: aggregate volume remains nearly flat while the agentic component changes.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
axes[0].plot(evaluation['minute'], evaluation['baseline_capacity'], label='Baseline capacity', linewidth=2, color='#524ae9')
axes[0].plot(evaluation['minute'], evaluation['aware_capacity_total'], label='Aware total capacity', linewidth=2, color='#fc5000')
axes[0].axvspan(60, 75, color='#f5f28e', alpha=.7, label='Composition shift')
axes[0].set_ylabel('Capacity units')
axes[0].set_title('Capacity response during the shift')
axes[0].legend()

axes[1].plot(evaluation['minute'], evaluation['baseline_projected_spend'], label='Baseline projected spend', color='#524ae9')
axes[1].plot(evaluation['minute'], evaluation['aware_projected_spend'], label='Aware projected spend', color='#fc5000')
axes[1].axhline(24.0, linestyle='--', color='#070607', label='Simulation budget')
axes[1].axvspan(60, 75, color='#f5f28e', alpha=.7)
axes[1].set_xlabel('Minute')
axes[1].set_ylabel('Projected spend')
axes[1].set_title('Cost-governance projection')
axes[1].legend()
plt.tight_layout()
plt.show()

display(evaluation.loc[evaluation['minute'].between(60, 75), ['minute', 'baseline_capacity', 'aware_capacity_total', 'baseline_projected_spend', 'aware_projected_spend']].head(16))

## 4. Sensitivity analysis

The sensitivity artifact varies agentic share and the service-time multiplier. Higher composition pressure should make the benefit of archetype-aware control more visible.

In [ ]:
pivot = sensitivity.pivot_table(index='agentic_share', columns='sla_multiplier', values='difference_baseline_minus_aware', aggfunc='median')
plt.figure(figsize=(10, 5))
sns.heatmap(pivot, annot=True, fmt='.2f', cmap='YlOrRd', center=0)
plt.title('Median SLA violation-rate improvement: baseline minus aware')
plt.xlabel('SLA multiplier')
plt.ylabel('Agentic share')
plt.tight_layout()
plt.show()
display(sensitivity.groupby(['agentic_share', 'sla_multiplier'])['difference_baseline_minus_aware'].median().reset_index())

## 5. Complete artifact inventory

The cells below include every committed CSV and JSON result artifact. Compact experiment tables are rendered in full. Large per-request tables are also rendered in full and can be downloaded from Colab.

In [ ]:
csv_artifacts = sorted(DATA.glob('*.csv'))
json_artifacts = sorted(DATA.glob('*.json'))
artifact_inventory = pd.DataFrame({
    'artifact': [p.name for p in csv_artifacts + json_artifacts],
    'format': [p.suffix.lstrip('.') for p in csv_artifacts + json_artifacts],
    'bytes': [p.stat().st_size for p in csv_artifacts + json_artifacts],
})
display(artifact_inventory)

for frame, name in [(evaluation, 'evaluation_results.csv'), (multiseed, 'multiseed_results.csv'), (sensitivity, 'sensitivity.csv')]:
    print(f'Complete table: {name} ({len(frame):,} rows)')
    display(frame)

rate25 = pd.read_csv(DATA / 'multiseed_results_rate25.csv')
print(f'Complete table: multiseed_results_rate25.csv ({len(rate25):,} rows)')
display(rate25)

## 6. Evaluation trace: capacity, spend, budget actions, and breaches

These graphs show all evaluation-series signals, not only the headline capacity and projected-spend lines.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(15, 14), sharex=True)
axes[0].plot(evaluation['minute'], evaluation['baseline_capacity'], label='Baseline capacity', color='#524ae9', linewidth=2)
axes[0].plot(evaluation['minute'], evaluation['aware_capacity_total'], label='Aware total capacity', color='#fc5000', linewidth=2)
for column, label, color in [('aware_capacity_bedrock_on_demand', 'Aware Bedrock on-demand', '#168aad'), ('aware_capacity_sagemaker_endpoint', 'Aware SageMaker endpoint', '#76c893'), ('aware_capacity_eks_gpu_reserved', 'Aware EKS GPU reserved', '#f4a261'), ('aware_capacity_eks_gpu_spot', 'Aware EKS GPU spot', '#e76f51')]:
    axes[0].plot(evaluation['minute'], evaluation[column], label=label, alpha=.8)
axes[0].set_ylabel('Capacity')
axes[0].set_title('All capacity signals')
axes[0].legend(ncol=3)

axes[1].plot(evaluation['minute'], evaluation['baseline_projected_spend'], label='Baseline projected spend', color='#524ae9')
axes[1].plot(evaluation['minute'], evaluation['aware_projected_spend'], label='Aware projected spend', color='#fc5000')
axes[1].plot(evaluation['minute'], evaluation['baseline_actual_spend'], label='Baseline actual spend', color='#7b2cbf', linestyle='--')
axes[1].plot(evaluation['minute'], evaluation['cumulative_actual_spend'], label='Cumulative actual spend', color='#2a9d8f', linestyle='--')
axes[1].axhline(24.0, linestyle=':', color='#070607', label='Simulation budget')
axes[1].set_ylabel('Spend')
axes[1].set_title('Projected and actual spend')
axes[1].legend(ncol=3)

axes[2].step(evaluation['minute'], evaluation['baseline_breach_projected'].astype(int), where='post', label='Baseline breach projected', color='#524ae9')
axes[2].step(evaluation['minute'], evaluation['aware_breach_projected'].astype(int), where='post', label='Aware breach projected', color='#fc5000')
axes[2].plot(evaluation['minute'], evaluation['baseline_budget_action'].map({'none': 0, 'scale_down': 1, 'throttle': 2}).fillna(-1), 'o-', label='Baseline budget action')
axes[2].plot(evaluation['minute'], evaluation['aware_budget_action'].map({'none': 0, 'scale_down': 1, 'throttle': 2}).fillna(-1), 'o-', label='Aware budget action')
axes[2].set_yticks([0, 1, 2])
axes[2].set_yticklabels(['none / false', 'scale_down / true', 'throttle'])
axes[2].set_xlabel('Minute')
axes[2].set_title('Budget projections, breaches, and actions')
axes[2].legend(ncol=2)
plt.tight_layout()
plt.show()

print('Complete evaluation table')
display(evaluation)

## 7. Full multi-seed results and alternative rate scenario

In [ ]:
print('Full primary multiseed table')
display(multiseed)
print('Full rate25 multiseed table')
display(rate25)

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
for frame, label, color in [(multiseed, 'primary', '#fc5000'), (rate25, 'rate25', '#524ae9')]:
    for sizing, marker in [('round', 'o'), ('ceil', 's')]:
        subset = frame[frame['sizing'].eq(sizing)]
        if len(subset):
            axes[0, 0].scatter(subset['baseline_p95_response_s'], subset['aware_p95_response_s'], label=f'{label} {sizing}', alpha=.75, marker=marker, color=color)
axes[0, 0].set_title('p95 response: aware vs baseline')
axes[0, 0].set_xlabel('Baseline p95 response (s)')
axes[0, 0].set_ylabel('Aware p95 response (s)')
axes[0, 0].legend()

long = pd.concat([multiseed.assign(scenario='primary'), rate25.assign(scenario='rate25')], ignore_index=True)
long_sla = long.melt(id_vars=['seed', 'sizing', 'scenario'], value_vars=['baseline_sla_violation_rate', 'aware_sla_violation_rate'], var_name='policy', value_name='rate')
sns.boxplot(data=long_sla, x='sizing', y='rate', hue='policy', ax=axes[0, 1])
axes[0, 1].set_title('SLA violation rate across all seeds')
axes[0, 1].set_ylabel('Violation rate')

long_spend = long.melt(id_vars=['seed', 'sizing', 'scenario'], value_vars=['baseline_actual_spend', 'aware_actual_spend'], var_name='policy', value_name='spend')
sns.boxplot(data=long_spend, x='sizing', y='spend', hue='policy', ax=axes[1, 0])
axes[1, 0].set_title('Actual spend across all seeds')
axes[1, 0].set_ylabel('Spend')

for frame, label, color in [(multiseed, 'primary', '#fc5000'), (rate25, 'rate25', '#524ae9')]:
    for sizing, marker in [('round', 'o'), ('ceil', 's')]:
        subset=frame[frame['sizing'].eq(sizing)]
        if len(subset):
            axes[1, 1].scatter(subset['aware_scaleup_minute'], subset['governance_lead_min'], label=f'{label} {sizing}', alpha=.75, marker=marker, color=color)
axes[1, 1].set_title('Aware scale-up timing and governance lead')
axes[1, 1].set_xlabel('Aware scale-up minute')
axes[1, 1].set_ylabel('Governance lead (min)')
axes[1, 1].legend()
plt.tight_layout()
plt.show()

## 8. Nested latency results by policy, archetype, and time window

In [ ]:
latency_rows = []
for sizing, sizing_result in latency.items():
    for policy in ['baseline', 'aware']:
        for grouping in ['overall', 'by_archetype', 'by_window']:
            values = sizing_result[policy][grouping]
            if grouping == 'overall':
                values = {'overall': values}
            for group, metrics in values.items():
                row = {'sizing': sizing, 'policy': policy, 'grouping': grouping, 'group': group}
                row.update(metrics)
                latency_rows.append(row)
latency_table = pd.DataFrame(latency_rows)
display(latency_table)

for grouping, title in [('by_archetype', 'Latency by archetype'), ('by_window', 'Latency by time window')]:
    subset = latency_table[latency_table['grouping'].eq(grouping)]
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    sns.barplot(data=subset, x='group', y='p95_response_s', hue='policy', col=None, ax=axes[0])
    axes[0].set_title(f'{title}: p95 response')
    axes[0].set_ylabel('p95 response (s)')
    axes[0].tick_params(axis='x', rotation=25)
    sns.barplot(data=subset, x='group', y='sla_violation_rate', hue='policy', ax=axes[1])
    axes[1].set_title(f'{title}: SLA violation rate')
    axes[1].set_ylabel('Violation rate')
    axes[1].tick_params(axis='x', rotation=25)
    plt.tight_layout()
    plt.show()

## 9. Per-request latency artifacts

All four per-request latency CSVs are loaded. The tables are complete, and the graphs compare response-time distributions and SLA outcomes by true archetype.

In [ ]:
per_request_frames = {}
for path in sorted(DATA.glob('latency_per_request_*.csv')):
    frame = pd.read_csv(path)
    per_request_frames[path.stem] = frame
    print(f'Complete table: {path.name} ({len(frame):,} rows)')
    display(frame)

plot_frames = []
for name, frame in per_request_frames.items():
    sizing = 'ceil' if name.endswith('_ceil') else 'round'
    policy = 'aware' if '_aware_' in name else 'baseline'
    selected = frame[['true_archetype', 'response_s', 'sla_violated']].copy()
    selected['sizing'] = sizing
    selected['policy'] = policy
    plot_frames.append(selected)
per_request_plot = pd.concat(plot_frames, ignore_index=True)
fig, axes = plt.subplots(1, 2, figsize=(17, 6))
sns.boxplot(data=per_request_plot, x='true_archetype', y='response_s', hue='policy', ax=axes[0], showfliers=False)
axes[0].set_title('Per-request response distribution')
axes[0].set_ylabel('Response time (s)')
axes[0].tick_params(axis='x', rotation=25)
sla_rates = per_request_plot.groupby(['sizing', 'policy', 'true_archetype'], as_index=False)['sla_violated'].mean()
sns.barplot(data=sla_rates, x='true_archetype', y='sla_violated', hue='policy', ax=axes[1])
axes[1].set_title('Per-request SLA violation rate')
axes[1].set_ylabel('Violation rate')
axes[1].tick_params(axis='x', rotation=25)
plt.tight_layout()
plt.show()

display(per_request_plot.groupby(['sizing', 'policy', 'true_archetype']).agg(requests=('response_s', 'size'), mean_response_s=('response_s', 'mean'), p50_response_s=('response_s', 'median'), p95_response_s=('response_s', lambda x: x.quantile(.95)), sla_violation_rate=('sla_violated', 'mean')).reset_index())

## 10. Trace and classifier artifacts

In [ ]:
print('Complete trace table')
display(pd.read_csv(DATA / 'trace.csv'))
print('Complete labeled trace table')
labeled_trace = pd.read_csv(DATA / 'trace_labeled.csv')
display(labeled_trace)

confusion = pd.crosstab(labeled_trace['true_archetype'], labeled_trace['predicted_archetype'], normalize='index')
plt.figure(figsize=(8, 6))
sns.heatmap(confusion, annot=True, fmt='.2f', cmap='YlOrRd', vmin=0, vmax=1)
plt.title('Classifier normalized confusion matrix')
plt.xlabel('Predicted archetype')
plt.ylabel('True archetype')
plt.tight_layout()
plt.show()

feature_columns = ['declared_output_len', 'prompt_domain_score', 'confidence', 'compute_cost']
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, column in zip(axes.flat, feature_columns):
    sns.histplot(data=labeled_trace, x=column, hue='predicted_archetype', element='step', stat='density', common_norm=False, ax=ax)
    ax.set_title(column)
plt.tight_layout()
plt.show()

## 11. Sensitivity tables and heatmaps for every sizing rule

In [ ]:
print('Complete sensitivity table')
display(sensitivity)
for sizing, subset in sensitivity.groupby('sizing'):
    pivot = subset.pivot_table(index='agentic_share', columns='sla_multiplier', values='difference_baseline_minus_aware', aggfunc='median')
    plt.figure(figsize=(10, 5))
    sns.heatmap(pivot, annot=True, fmt='.2f', cmap='YlOrRd', center=0)
    plt.title(f'Sensitivity: baseline minus aware SLA violation rate ({sizing})')
    plt.xlabel('SLA multiplier')
    plt.ylabel('Agentic share')
    plt.tight_layout()
    plt.show()

print('Sensitivity medians by scenario')
display(sensitivity.groupby(['sizing', 'agentic_share', 'sla_multiplier'])['difference_baseline_minus_aware'].median().reset_index())

## 12. Reproducibility and limitations

Run the repository acceptance suite separately from this presentation notebook:

```bash
pytest -q
mypy src/
```

Important limitations: serving pools are emulated, the live AWS layer is separate from the core simulation, the trace family is synthetic, SLA values were fixed before evaluation, and benchmark figures should be interpreted as measured project evidence rather than production capacity guarantees.